# DeblurGAN License Plate Restoration Workflow

This notebook implements the deblurring stage after the YOLOv8 detector. It does not edit or depend on modifying `yolov8_workflow.ipynb`.

Pipeline in this notebook:

1. Load the trained YOLOv8 plate detector from `runs/bangla_plate_yolov8/plate_motion_blur_distortion_exp/weights/best.pt`.
2. Build DeblurGAN training pairs where the input is a YOLOv8-cropped blurry license plate and the target is the matching clean plate crop.
3. Train a DeblurGAN model with an FPN generator using an Inception-ResNet-V2 backbone, a relativistic double-scale PatchGAN discriminator, RaGAN-LS adversarial loss, pixel loss, and perceptual loss.
4. Save checkpoints and run deblur inference on YOLO-cropped plate images.


In [ ]:
# Optional: install timm to use the full timm Inception-ResNet-V2 backbone instead of the notebook-local fallback.
# The rest of the notebook uses packages already needed by the YOLOv8 workflow.
# %pip install -q timm


In [ ]:
from pathlib import Path
import gc
import math
import random
import shutil
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from ultralytics import YOLO

try:
    from tqdm.auto import tqdm
except ImportError:
    class _SimpleTqdm:
        def __init__(self, iterable=None, *args, **kwargs):
            self.iterable = iterable if iterable is not None else []

        def __iter__(self):
            return iter(self.iterable)

        def set_postfix(self, *args, **kwargs):
            pass

        def close(self):
            pass

        def __enter__(self):
            return self

        def __exit__(self, exc_type, exc_value, traceback):
            self.close()

    def tqdm(iterable=None, *args, **kwargs):
        return _SimpleTqdm(iterable, *args, **kwargs)

    print('tqdm is not installed. Progress bars are disabled, but the notebook will still run.')

try:
    import timm
except ImportError:
    timm = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TORCH_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
YOLO_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
# Crop extraction defaults to CPU to keep GPU memory free for DeblurGAN training.
# Set this to 0 only if you have enough free VRAM and want faster YOLO cropping.
YOLO_CROP_DEVICE = 'cpu'

PROJECT_ROOT = Path('.')
PLATE_DATASET = PROJECT_ROOT / 'dataset_plate'
YOLO_WEIGHTS = PROJECT_ROOT / 'runs' / 'bangla_plate_yolov8' / 'plate_motion_blur_distortion_exp' / 'weights' / 'best.pt'

DEBLUR_DATA_ROOT = PROJECT_ROOT / 'deblurgan_data'
YOLO_HANDOFF_ROOT = DEBLUR_DATA_ROOT / 'yolov8_handoff'
BLURRY_FULL_ROOT = YOLO_HANDOFF_ROOT / 'blurred_full_images'
PAIR_ROOT = DEBLUR_DATA_ROOT / 'paired_plate_crops'
RUN_ROOT = PROJECT_ROOT / 'runs' / 'deblurgan_irv2_fpn'
CHECKPOINT_ROOT = RUN_ROOT / 'checkpoints'

SPLITS = ['train', 'valid', 'test']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

# Keep None for full data, or set smaller numbers for a quick smoke run.
MAX_ITEMS_PER_SPLIT = {
    'train': None,
    'valid': None,
    'test': None,
}

PLATE_HEIGHT = 96
PLATE_WIDTH = 256
YOLO_IMAGE_SIZE = 640
YOLO_CONF = 0.25
YOLO_CROP_BATCH_SIZE = 1
CROP_PAD_RATIO = 0.08
REBUILD_DEBLUR_DATA = False

BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS = 20
LEARNING_RATE_G = 1e-4
LEARNING_RATE_D = 1e-4

LAMBDA_PIXEL = 100.0
LAMBDA_PERCEPTUAL = 10.0
LAMBDA_ADV = 1.0

BACKBONE_PRETRAINED = False
USE_PRETRAINED_VGG = True

print('Torch device:', TORCH_DEVICE)
print('YOLO weights:', YOLO_WEIGHTS.resolve())
print('Deblur data root:', DEBLUR_DATA_ROOT.resolve())


In [7]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = 0
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    DEVICE = "cpu"
    print("GPU not detected. Training will run on CPU.")

print("Training device:", DEVICE)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: NVIDIA GeForce RTX 3060
CUDA version: 12.8
Training device: 0


## YOLOv8 Crop Handoff

The input to DeblurGAN is created by YOLOv8 itself. The notebook first creates blurry full-image copies from the clean plate dataset, then runs the trained YOLOv8 detector on those blurry images. The predicted YOLO box is used to crop both the blurry input and the matching clean target.


In [ ]:
rng = np.random.default_rng(SEED)

def list_images(image_dir):
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)


def make_motion_kernel(size, angle):
    kernel = np.zeros((size, size), dtype=np.float32)
    center = size // 2
    cv2.line(kernel, (0, center), (size - 1, center), 1.0, 1)
    matrix = cv2.getRotationMatrix2D((center, center), angle, 1.0)
    kernel = cv2.warpAffine(kernel, matrix, (size, size))
    kernel_sum = kernel.sum()
    if kernel_sum <= 0:
        kernel[center, center] = 1.0
        kernel_sum = 1.0
    return kernel / kernel_sum


def degrade_full_image(image_bgr):
    degraded = image_bgr.copy()

    if rng.random() < 0.85:
        kernel_size = int(rng.choice([7, 9, 11, 13, 15, 17, 21, 25]))
        angle = float(rng.uniform(-25.0, 25.0))
        degraded = cv2.filter2D(degraded, -1, make_motion_kernel(kernel_size, angle))

    if rng.random() < 0.35:
        blur_size = int(rng.choice([3, 5, 7]))
        degraded = cv2.GaussianBlur(degraded, (blur_size, blur_size), 0)

    if rng.random() < 0.50:
        alpha = float(rng.uniform(0.75, 1.25))
        beta = float(rng.uniform(-28.0, 28.0))
        degraded = cv2.convertScaleAbs(degraded, alpha=alpha, beta=beta)

    if rng.random() < 0.35:
        noise = rng.normal(0.0, rng.uniform(3.0, 12.0), degraded.shape).astype(np.float32)
        degraded = np.clip(degraded.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if rng.random() < 0.35:
        quality = int(rng.integers(35, 85))
        ok, encoded = cv2.imencode('.jpg', degraded, [cv2.IMWRITE_JPEG_QUALITY, quality])
        if ok:
            degraded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)

    return degraded


def pad_and_clip_xyxy(box, width, height, pad_ratio):
    x1, y1, x2, y2 = [float(v) for v in box]
    box_width = max(1.0, x2 - x1)
    box_height = max(1.0, y2 - y1)
    pad_x = box_width * pad_ratio
    pad_y = box_height * pad_ratio
    x1 = max(0, int(round(x1 - pad_x)))
    y1 = max(0, int(round(y1 - pad_y)))
    x2 = min(width, int(round(x2 + pad_x)))
    y2 = min(height, int(round(y2 + pad_y)))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def crop_resize(image_bgr, box, output_size):
    height, width = image_bgr.shape[:2]
    clipped = pad_and_clip_xyxy(box, width, height, CROP_PAD_RATIO)
    if clipped is None:
        return None
    x1, y1, x2, y2 = clipped
    crop = image_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, (output_size[1], output_size[0]), interpolation=cv2.INTER_AREA)


def best_yolo_box(result):
    if result.boxes is None or len(result.boxes) == 0:
        return None
    confidences = result.boxes.conf.detach().cpu().numpy()
    best_index = int(confidences.argmax())
    return result.boxes.xyxy[best_index].detach().cpu().numpy().tolist()


def build_yolov8_deblur_pairs():
    if not YOLO_WEIGHTS.exists():
        raise FileNotFoundError(f'Missing YOLO weights: {YOLO_WEIGHTS}')
    if not PLATE_DATASET.exists():
        raise FileNotFoundError(f'Missing plate dataset: {PLATE_DATASET}')

    if REBUILD_DEBLUR_DATA and DEBLUR_DATA_ROOT.exists():
        shutil.rmtree(DEBLUR_DATA_ROOT)

    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(YOLO_WEIGHTS))
    crop_device = globals().get('YOLO_CROP_DEVICE', 'cpu')
    crop_batch_size = globals().get('YOLO_CROP_BATCH_SIZE', 1)
    manifests = {}

    for split in SPLITS:
        manifest_path = PAIR_ROOT / split / 'manifest.csv'
        if manifest_path.exists() and not REBUILD_DEBLUR_DATA:
            manifests[split] = manifest_path
            print(f'{split}: using existing manifest at {manifest_path}')
            continue

        source_image_dir = PLATE_DATASET / split / 'images'
        source_label_dir = PLATE_DATASET / split / 'labels'
        blurry_full_dir = BLURRY_FULL_ROOT / split
        blurry_crop_dir = PAIR_ROOT / split / 'blurry'
        sharp_crop_dir = PAIR_ROOT / split / 'sharp'

        blurry_full_dir.mkdir(parents=True, exist_ok=True)
        blurry_crop_dir.mkdir(parents=True, exist_ok=True)
        sharp_crop_dir.mkdir(parents=True, exist_ok=True)

        image_paths = list_images(source_image_dir)
        limit = MAX_ITEMS_PER_SPLIT.get(split)
        if limit is not None:
            image_paths = image_paths[:limit]

        handoff_records = []
        for image_path in tqdm(image_paths, desc=f'{split}: creating blurry full images'):
            label_path = source_label_dir / f'{image_path.stem}.txt'
            if not label_path.exists() or not label_path.read_text(encoding='utf-8').strip():
                continue

            clean_image = cv2.imread(str(image_path))
            if clean_image is None:
                continue

            blurry_image = degrade_full_image(clean_image)
            blurry_full_path = blurry_full_dir / image_path.name
            cv2.imwrite(str(blurry_full_path), blurry_image)
            handoff_records.append((blurry_full_path, image_path))

        rows = []
        for blurry_full_path, clean_path in tqdm(handoff_records, total=len(handoff_records), desc=f'{split}: YOLO crops'):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            result = model.predict(
                source=str(blurry_full_path),
                imgsz=YOLO_IMAGE_SIZE,
                conf=YOLO_CONF,
                device=crop_device,
                batch=crop_batch_size,
                half=False,
                verbose=False,
                stream=False,
            )[0]
            box = best_yolo_box(result)
            if box is None:
                continue

            blurry_image = cv2.imread(str(blurry_full_path))
            clean_image = cv2.imread(str(clean_path))
            if blurry_image is None or clean_image is None:
                continue

            blurry_crop = crop_resize(blurry_image, box, (PLATE_HEIGHT, PLATE_WIDTH))
            sharp_crop = crop_resize(clean_image, box, (PLATE_HEIGHT, PLATE_WIDTH))
            if blurry_crop is None or sharp_crop is None:
                continue

            output_name = f'{clean_path.stem}.png'
            blurry_crop_path = blurry_crop_dir / output_name
            sharp_crop_path = sharp_crop_dir / output_name
            cv2.imwrite(str(blurry_crop_path), blurry_crop)
            cv2.imwrite(str(sharp_crop_path), sharp_crop)

            rows.append({
                'blurry': str(blurry_crop_path),
                'sharp': str(sharp_crop_path),
                'source_image': str(clean_path),
                'blurry_full_image': str(blurry_full_path),
                'x1': box[0],
                'y1': box[1],
                'x2': box[2],
                'y2': box[3],
            })

            del result
            gc.collect()

        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(manifest_path, index=False)
        manifests[split] = manifest_path
        print(f'{split}: wrote {len(rows)} YOLO crop pairs to {manifest_path}')

    return manifests


manifests = build_yolov8_deblur_pairs()
for split, manifest in manifests.items():
    count = len(pd.read_csv(manifest)) if manifest.exists() else 0
    print(f'{split}: {count} paired YOLO-crop samples')


## Dataset Loaders


In [ ]:
class PlateDeblurDataset(Dataset):
    def __init__(self, manifest_path):
        self.frame = pd.read_csv(manifest_path)
        if len(self.frame) == 0:
            raise ValueError(f'No samples found in {manifest_path}')

    def __len__(self):
        return len(self.frame)

    def _read_image(self, path):
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f'Could not read image: {path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (PLATE_WIDTH, PLATE_HEIGHT), interpolation=cv2.INTER_AREA)
        tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        return tensor

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        blurry = self._read_image(row['blurry'])
        sharp = self._read_image(row['sharp'])
        return blurry, sharp


train_dataset = PlateDeblurDataset(manifests['train'])
valid_dataset = PlateDeblurDataset(manifests['valid'])
test_dataset = PlateDeblurDataset(manifests['test'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print('Train samples:', len(train_dataset))
print('Valid samples:', len(valid_dataset))
print('Test samples:', len(test_dataset))

blurry_batch, sharp_batch = next(iter(train_loader))
print('Batch input shape:', tuple(blurry_batch.shape))
print('Batch target shape:', tuple(sharp_batch.shape))


## Model Architecture and Losses


In [ ]:
def group_norm(channels):
    groups = 8 if channels % 8 == 0 else 1
    return nn.GroupNorm(groups, channels)


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class InceptionResNetA(nn.Module):
    def __init__(self, channels, scale=0.17):
        super().__init__()
        self.scale = scale
        self.branch_1 = ConvBlock(channels, 32, kernel_size=1)
        self.branch_2 = nn.Sequential(ConvBlock(channels, 32, kernel_size=1), ConvBlock(32, 32, kernel_size=3))
        self.branch_3 = nn.Sequential(ConvBlock(channels, 32, kernel_size=1), ConvBlock(32, 48, kernel_size=3), ConvBlock(48, 64, kernel_size=3))
        self.projection = nn.Conv2d(128, channels, kernel_size=1)
        self.activation = nn.SiLU(inplace=True)

    def forward(self, x):
        residual = torch.cat([self.branch_1(x), self.branch_2(x), self.branch_3(x)], dim=1)
        return self.activation(x + self.scale * self.projection(residual))


class InceptionResNetB(nn.Module):
    def __init__(self, channels, scale=0.10):
        super().__init__()
        self.scale = scale
        self.branch_1 = ConvBlock(channels, 96, kernel_size=1)
        self.branch_2 = nn.Sequential(
            ConvBlock(channels, 64, kernel_size=1),
            nn.Conv2d(64, 80, kernel_size=(1, 7), padding=(0, 3), bias=False),
            group_norm(80),
            nn.SiLU(inplace=True),
            nn.Conv2d(80, 96, kernel_size=(7, 1), padding=(3, 0), bias=False),
            group_norm(96),
            nn.SiLU(inplace=True),
        )
        self.projection = nn.Conv2d(192, channels, kernel_size=1)
        self.activation = nn.SiLU(inplace=True)

    def forward(self, x):
        residual = torch.cat([self.branch_1(x), self.branch_2(x)], dim=1)
        return self.activation(x + self.scale * self.projection(residual))


class InceptionResNetC(nn.Module):
    def __init__(self, channels, scale=0.20):
        super().__init__()
        self.scale = scale
        self.branch_1 = ConvBlock(channels, 96, kernel_size=1)
        self.branch_2 = nn.Sequential(
            ConvBlock(channels, 64, kernel_size=1),
            nn.Conv2d(64, 96, kernel_size=(1, 3), padding=(0, 1), bias=False),
            group_norm(96),
            nn.SiLU(inplace=True),
            nn.Conv2d(96, 96, kernel_size=(3, 1), padding=(1, 0), bias=False),
            group_norm(96),
            nn.SiLU(inplace=True),
        )
        self.projection = nn.Conv2d(192, channels, kernel_size=1)
        self.activation = nn.SiLU(inplace=True)

    def forward(self, x):
        residual = torch.cat([self.branch_1(x), self.branch_2(x)], dim=1)
        return self.activation(x + self.scale * self.projection(residual))


class ReductionA(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.branch_1 = ConvBlock(in_channels, 192, kernel_size=3, stride=2)
        self.branch_2 = nn.Sequential(ConvBlock(in_channels, 128, kernel_size=1), ConvBlock(128, 160, kernel_size=3), ConvBlock(160, 192, kernel_size=3, stride=2))
        self.branch_3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.projection = ConvBlock(in_channels + 384, out_channels, kernel_size=1)

    def forward(self, x):
        merged = torch.cat([self.branch_1(x), self.branch_2(x), self.branch_3(x)], dim=1)
        return self.projection(merged)


class ReductionB(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.branch_1 = nn.Sequential(ConvBlock(in_channels, 192, kernel_size=1), ConvBlock(192, 256, kernel_size=3, stride=2))
        self.branch_2 = nn.Sequential(ConvBlock(in_channels, 192, kernel_size=1), ConvBlock(192, 224, kernel_size=3), ConvBlock(224, 256, kernel_size=3, stride=2))
        self.branch_3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.projection = ConvBlock(in_channels + 512, out_channels, kernel_size=1)

    def forward(self, x):
        merged = torch.cat([self.branch_1(x), self.branch_2(x), self.branch_3(x)], dim=1)
        return self.projection(merged)


class LocalInceptionResNetV2Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_channels = [32, 64, 192, 384, 768]
        self.stem_1 = ConvBlock(3, 32, kernel_size=3, stride=2)
        self.stem_2 = ConvBlock(32, 64, kernel_size=3, stride=2)
        self.stem_3 = nn.Sequential(ConvBlock(64, 128, kernel_size=3, stride=2), ConvBlock(128, 192, kernel_size=3))
        self.block_a = nn.Sequential(InceptionResNetA(192), InceptionResNetA(192), InceptionResNetA(192))
        self.reduction_a = ReductionA(192, 384)
        self.block_b = nn.Sequential(InceptionResNetB(384), InceptionResNetB(384), InceptionResNetB(384))
        self.reduction_b = ReductionB(384, 768)
        self.block_c = nn.Sequential(InceptionResNetC(768), InceptionResNetC(768))

    def forward(self, x):
        f0 = self.stem_1(x)
        f1 = self.stem_2(f0)
        f2 = self.stem_3(f1)
        f2 = self.block_a(f2)
        f3 = self.reduction_a(f2)
        f3 = self.block_b(f3)
        f4 = self.reduction_b(f3)
        f4 = self.block_c(f4)
        return [f0, f1, f2, f3, f4]


class FPNInceptionResNetV2Generator(nn.Module):
    def __init__(self, fpn_channels=128, pretrained_backbone=False):
        super().__init__()
        if timm is not None:
            self.backbone = timm.create_model(
                'inception_resnet_v2',
                pretrained=pretrained_backbone,
                features_only=True,
                out_indices=(0, 1, 2, 3, 4),
            )
            channels = self.backbone.feature_info.channels()
        else:
            print('timm is not available. Using the notebook-local Inception-ResNet-V2-style backbone.')
            self.backbone = LocalInceptionResNetV2Backbone()
            channels = self.backbone.feature_channels
        self.lateral = nn.ModuleList([nn.Conv2d(channel, fpn_channels, kernel_size=1) for channel in channels])
        self.smooth = nn.ModuleList([ConvBlock(fpn_channels, fpn_channels) for _ in channels])
        self.decoder = nn.Sequential(
            ConvBlock(fpn_channels * len(channels), fpn_channels),
            ConvBlock(fpn_channels, fpn_channels // 2),
            ConvBlock(fpn_channels // 2, fpn_channels // 4),
            nn.Conv2d(fpn_channels // 4, 3, kernel_size=3, padding=1),
        )

    def forward(self, x):
        input_height, input_width = x.shape[-2:]
        features = self.backbone(x)
        pyramid = [None] * len(features)
        top = self.lateral[-1](features[-1])
        pyramid[-1] = self.smooth[-1](top)

        for index in range(len(features) - 2, -1, -1):
            top = F.interpolate(top, size=features[index].shape[-2:], mode='bilinear', align_corners=False)
            top = top + self.lateral[index](features[index])
            pyramid[index] = self.smooth[index](top)

        fused = torch.cat(
            [F.interpolate(level, size=(input_height, input_width), mode='bilinear', align_corners=False) for level in pyramid],
            dim=1,
        )
        residual = torch.tanh(self.decoder(fused)) * 0.5
        return torch.clamp(x + residual, 0.0, 1.0)


class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, layers=4):
        super().__init__()
        blocks = [
            nn.Conv2d(in_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        channels = base_channels
        for layer_index in range(1, layers):
            next_channels = min(base_channels * (2 ** layer_index), 512)
            blocks.extend([
                nn.Conv2d(channels, next_channels, kernel_size=4, stride=2, padding=1, bias=False),
                group_norm(next_channels),
                nn.LeakyReLU(0.2, inplace=True),
            ])
            channels = next_channels
        blocks.extend([
            nn.Conv2d(channels, channels, kernel_size=4, stride=1, padding=1, bias=False),
            group_norm(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, 1, kernel_size=4, stride=1, padding=1),
        ])
        self.model = nn.Sequential(*blocks)

    def forward(self, x):
        return self.model(x)


class DoubleScalePatchGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.full_scale = PatchDiscriminator()
        self.half_scale = PatchDiscriminator()

    def forward(self, x):
        half = F.interpolate(x, scale_factor=0.5, mode='bilinear', align_corners=False, recompute_scale_factor=False)
        return [self.full_scale(x), self.half_scale(half)]


def ragan_ls_discriminator_loss(real_outputs, fake_outputs):
    total = 0.0
    for real, fake in zip(real_outputs, fake_outputs):
        real_mean = real.mean()
        fake_mean = fake.mean()
        total = total + 0.5 * ((real - fake_mean - 1.0) ** 2).mean()
        total = total + 0.5 * ((fake - real_mean + 1.0) ** 2).mean()
    return total / len(real_outputs)


def ragan_ls_generator_loss(real_outputs, fake_outputs):
    total = 0.0
    for real, fake in zip(real_outputs, fake_outputs):
        real_mean = real.mean()
        fake_mean = fake.mean()
        total = total + 0.5 * ((real - fake_mean + 1.0) ** 2).mean()
        total = total + 0.5 * ((fake - real_mean - 1.0) ** 2).mean()
    return total / len(real_outputs)


class VGGPerceptualLoss(nn.Module):
    def __init__(self, use_pretrained=True):
        super().__init__()
        try:
            weights = models.VGG16_Weights.DEFAULT if use_pretrained else None
            vgg = models.vgg16(weights=weights).features[:16]
        except Exception as exc:
            print('Pretrained VGG16 could not be loaded. Falling back to random VGG16 features.')
            print(exc)
            vgg = models.vgg16(weights=None).features[:16]

        for parameter in vgg.parameters():
            parameter.requires_grad = False
        self.features = vgg.eval()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, generated, target):
        generated = (generated - self.mean) / self.std
        target = (target - self.mean) / self.std
        return F.l1_loss(self.features(generated), self.features(target))


def set_requires_grad(module, requires_grad):
    for parameter in module.parameters():
        parameter.requires_grad = requires_grad


generator = FPNInceptionResNetV2Generator(pretrained_backbone=BACKBONE_PRETRAINED).to(TORCH_DEVICE)
discriminator = DoubleScalePatchGANDiscriminator().to(TORCH_DEVICE)
pixel_loss_fn = nn.L1Loss().to(TORCH_DEVICE)
perceptual_loss_fn = VGGPerceptualLoss(use_pretrained=USE_PRETRAINED_VGG).to(TORCH_DEVICE)

with torch.no_grad():
    sample_input = torch.rand(1, 3, PLATE_HEIGHT, PLATE_WIDTH, device=TORCH_DEVICE)
    sample_output = generator(sample_input)
    sample_disc = discriminator(sample_output)

print('Generator output shape:', tuple(sample_output.shape))
print('Discriminator patch shapes:', [tuple(output.shape) for output in sample_disc])


## Training Loop


In [ ]:
def batch_psnr(prediction, target):
    mse = F.mse_loss(prediction, target, reduction='none').flatten(1).mean(dim=1)
    return (10.0 * torch.log10(1.0 / (mse + 1e-8))).mean().item()


@torch.no_grad()
def validate(generator_model, loader):
    generator_model.eval()
    total_l1 = 0.0
    total_psnr = 0.0
    total_count = 0
    for blurry, sharp in loader:
        blurry = blurry.to(TORCH_DEVICE, non_blocking=True)
        sharp = sharp.to(TORCH_DEVICE, non_blocking=True)
        restored = generator_model(blurry)
        batch_size = blurry.size(0)
        total_l1 += F.l1_loss(restored, sharp, reduction='mean').item() * batch_size
        total_psnr += batch_psnr(restored, sharp) * batch_size
        total_count += batch_size
    return {
        'l1': total_l1 / max(1, total_count),
        'psnr': total_psnr / max(1, total_count),
    }


optimizer_g = torch.optim.AdamW(generator.parameters(), lr=LEARNING_RATE_G, betas=(0.5, 0.999))
optimizer_d = torch.optim.AdamW(discriminator.parameters(), lr=LEARNING_RATE_D, betas=(0.5, 0.999))
history = []
best_valid_l1 = float('inf')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    generator.train()
    discriminator.train()
    epoch_start = time.time()
    running = {
        'd_loss': 0.0,
        'g_loss': 0.0,
        'pixel': 0.0,
        'perceptual': 0.0,
        'adv': 0.0,
    }
    steps = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for blurry, sharp in progress:
        blurry = blurry.to(TORCH_DEVICE, non_blocking=True)
        sharp = sharp.to(TORCH_DEVICE, non_blocking=True)

        set_requires_grad(discriminator, True)
        optimizer_d.zero_grad(set_to_none=True)
        with torch.no_grad():
            fake_detached = generator(blurry)
        real_outputs = discriminator(sharp)
        fake_outputs = discriminator(fake_detached.detach())
        d_loss = ragan_ls_discriminator_loss(real_outputs, fake_outputs)
        d_loss.backward()
        optimizer_d.step()

        set_requires_grad(discriminator, False)
        optimizer_g.zero_grad(set_to_none=True)
        restored = generator(blurry)
        fake_outputs = discriminator(restored)
        real_outputs = discriminator(sharp.detach())
        adv_loss = ragan_ls_generator_loss(real_outputs, fake_outputs)
        pixel_loss = pixel_loss_fn(restored, sharp)
        perceptual_loss = perceptual_loss_fn(restored, sharp)
        g_loss = LAMBDA_PIXEL * pixel_loss + LAMBDA_PERCEPTUAL * perceptual_loss + LAMBDA_ADV * adv_loss
        g_loss.backward()
        optimizer_g.step()

        running['d_loss'] += d_loss.item()
        running['g_loss'] += g_loss.item()
        running['pixel'] += pixel_loss.item()
        running['perceptual'] += perceptual_loss.item()
        running['adv'] += adv_loss.item()
        steps += 1

        progress.set_postfix({
            'D': running['d_loss'] / steps,
            'G': running['g_loss'] / steps,
            'L1': running['pixel'] / steps,
        })

    validation = validate(generator, valid_loader)
    row = {
        'epoch': epoch,
        'seconds': time.time() - epoch_start,
        'train_d_loss': running['d_loss'] / max(1, steps),
        'train_g_loss': running['g_loss'] / max(1, steps),
        'train_pixel_loss': running['pixel'] / max(1, steps),
        'train_perceptual_loss': running['perceptual'] / max(1, steps),
        'train_adv_loss': running['adv'] / max(1, steps),
        'valid_l1': validation['l1'],
        'valid_psnr': validation['psnr'],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_ROOT / 'training_history.csv', index=False)

    checkpoint = {
        'epoch': epoch,
        'generator': generator.state_dict(),
        'discriminator': discriminator.state_dict(),
        'optimizer_g': optimizer_g.state_dict(),
        'optimizer_d': optimizer_d.state_dict(),
        'config': {
            'plate_height': PLATE_HEIGHT,
            'plate_width': PLATE_WIDTH,
            'lambda_pixel': LAMBDA_PIXEL,
            'lambda_perceptual': LAMBDA_PERCEPTUAL,
            'lambda_adv': LAMBDA_ADV,
        },
    }
    torch.save(checkpoint, CHECKPOINT_ROOT / 'latest.pt')
    if validation['l1'] < best_valid_l1:
        best_valid_l1 = validation['l1']
        torch.save(checkpoint, CHECKPOINT_ROOT / 'best_generator.pt')

    valid_l1 = validation['l1']
    valid_psnr = validation['psnr']
    print(f'Epoch {epoch}: valid L1={valid_l1:.5f}, PSNR={valid_psnr:.2f} dB')

test_metrics = validate(generator, test_loader)
print('Final test metrics:', test_metrics)


## Visual Check


In [ ]:
def tensor_to_image(tensor):
    array = tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    return array


@torch.no_grad()
def show_deblur_examples(loader, count=4):
    generator.eval()
    blurry, sharp = next(iter(loader))
    blurry = blurry.to(TORCH_DEVICE)
    sharp = sharp.to(TORCH_DEVICE)
    restored = generator(blurry)
    count = min(count, blurry.size(0))

    plt.figure(figsize=(12, 3 * count))
    for index in range(count):
        plt.subplot(count, 3, index * 3 + 1)
        plt.imshow(tensor_to_image(blurry[index]))
        plt.title('YOLO blurry crop')
        plt.axis('off')
        plt.subplot(count, 3, index * 3 + 2)
        plt.imshow(tensor_to_image(restored[index]))
        plt.title('DeblurGAN output')
        plt.axis('off')
        plt.subplot(count, 3, index * 3 + 3)
        plt.imshow(tensor_to_image(sharp[index]))
        plt.title('Clean target')
        plt.axis('off')
    plt.tight_layout()
    plt.show()


show_deblur_examples(test_loader, count=4)
history_path = RUN_ROOT / 'training_history.csv'
if history_path.exists():
    history_frame = pd.read_csv(history_path)
    display(history_frame.tail())


## Inference from YOLOv8 Crops

Use this section after training. It supports both full images, where YOLOv8 is run first and its crop is passed into DeblurGAN, and existing folders of YOLO-cropped plate images.


In [ ]:
def load_trained_generator(checkpoint_path=CHECKPOINT_ROOT / 'best_generator.pt'):
    model = FPNInceptionResNetV2Generator(pretrained_backbone=False).to(TORCH_DEVICE)
    checkpoint = torch.load(checkpoint_path, map_location=TORCH_DEVICE)
    model.load_state_dict(checkpoint['generator'])
    model.eval()
    return model


def deblur_crop_array(generator_model, crop_bgr):
    resized = cv2.resize(crop_bgr, (PLATE_WIDTH, PLATE_HEIGHT), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    tensor = tensor.to(TORCH_DEVICE)
    with torch.no_grad():
        restored = generator_model(tensor)[0]
    restored_rgb = (restored.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(restored_rgb, cv2.COLOR_RGB2BGR)


def deblur_full_images_with_yolo(source, output_dir=RUN_ROOT / 'inference_from_full_images'):
    generator_model = load_trained_generator()
    yolo_model = YOLO(str(YOLO_WEIGHTS))
    output_dir = Path(output_dir)
    crop_dir = output_dir / 'yolov8_crops'
    deblur_dir = output_dir / 'deblurred'
    crop_dir.mkdir(parents=True, exist_ok=True)
    deblur_dir.mkdir(parents=True, exist_ok=True)

    results = yolo_model.predict(
        source=str(source),
        imgsz=YOLO_IMAGE_SIZE,
        conf=YOLO_CONF,
        device=YOLO_CROP_DEVICE,
        batch=YOLO_CROP_BATCH_SIZE,
        half=False,
        verbose=False,
        stream=True,
    )

    saved = []
    for result in results:
        image_path = Path(result.path)
        image = cv2.imread(str(image_path))
        if image is None:
            continue
        boxes = [] if result.boxes is None else result.boxes.xyxy.detach().cpu().numpy().tolist()
        for index, box in enumerate(boxes):
            crop = crop_resize(image, box, (PLATE_HEIGHT, PLATE_WIDTH))
            if crop is None:
                continue
            crop_path = crop_dir / f'{image_path.stem}_plate_{index}.png'
            output_path = deblur_dir / f'{image_path.stem}_plate_{index}_deblurred.png'
            restored = deblur_crop_array(generator_model, crop)
            cv2.imwrite(str(crop_path), crop)
            cv2.imwrite(str(output_path), restored)
            saved.append(output_path)

    print(f'Saved {len(saved)} deblurred plate crops to {deblur_dir}')
    return saved


def deblur_existing_yolo_crop_folder(crop_folder, output_dir=RUN_ROOT / 'inference_from_existing_yolo_crops'):
    generator_model = load_trained_generator()
    crop_folder = Path(crop_folder)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = []

    for crop_path in list_images(crop_folder):
        crop = cv2.imread(str(crop_path))
        if crop is None:
            continue
        restored = deblur_crop_array(generator_model, crop)
        output_path = output_dir / f'{crop_path.stem}_deblurred.png'
        cv2.imwrite(str(output_path), restored)
        saved.append(output_path)

    print(f'Saved {len(saved)} deblurred crops to {output_dir}')
    return saved


# Example after training:
# deblur_full_images_with_yolo(PLATE_DATASET / 'test' / 'images')
# deblur_existing_yolo_crop_folder(PAIR_ROOT / 'test' / 'blurry')


## DeblurGAN Evaluation Figures

This final section creates thesis-ready restoration figures for the deblurring stage: training curves, PSNR/SSIM/MAE comparison, and qualitative before-after-target examples.


In [ ]:
EVALUATION_ROOT = RUN_ROOT / 'evaluation_figures'
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)


def get_evaluation_generator():
    checkpoint_path = CHECKPOINT_ROOT / 'best_generator.pt'
    if checkpoint_path.exists():
        return load_trained_generator(checkpoint_path)
    if 'generator' in globals():
        generator.eval()
        return generator
    raise FileNotFoundError('No trained generator found. Run training first or provide best_generator.pt.')


def psnr_tensor(prediction, target):
    mse = F.mse_loss(prediction, target, reduction='none').flatten(1).mean(dim=1)
    return 10.0 * torch.log10(1.0 / (mse + 1e-8))


def ssim_tensor(prediction, target, window_size=7):
    channels = prediction.size(1)
    padding = window_size // 2
    window = torch.ones((channels, 1, window_size, window_size), device=prediction.device) / (window_size * window_size)

    mu_prediction = F.conv2d(prediction, window, padding=padding, groups=channels)
    mu_target = F.conv2d(target, window, padding=padding, groups=channels)
    mu_prediction_sq = mu_prediction.pow(2)
    mu_target_sq = mu_target.pow(2)
    mu_product = mu_prediction * mu_target

    sigma_prediction = F.conv2d(prediction * prediction, window, padding=padding, groups=channels) - mu_prediction_sq
    sigma_target = F.conv2d(target * target, window, padding=padding, groups=channels) - mu_target_sq
    sigma_product = F.conv2d(prediction * target, window, padding=padding, groups=channels) - mu_product

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    numerator = (2 * mu_product + c1) * (2 * sigma_product + c2)
    denominator = (mu_prediction_sq + mu_target_sq + c1) * (sigma_prediction + sigma_target + c2)
    return (numerator / (denominator + 1e-8)).flatten(1).mean(dim=1)


@torch.no_grad()
def collect_restoration_metrics(generator_model, loader):
    generator_model.eval()
    rows = []
    for blurry, sharp in loader:
        blurry = blurry.to(TORCH_DEVICE, non_blocking=True)
        sharp = sharp.to(TORCH_DEVICE, non_blocking=True)
        restored = generator_model(blurry)

        blurry_psnr = psnr_tensor(blurry, sharp)
        restored_psnr = psnr_tensor(restored, sharp)
        blurry_ssim = ssim_tensor(blurry, sharp)
        restored_ssim = ssim_tensor(restored, sharp)
        blurry_mae = F.l1_loss(blurry, sharp, reduction='none').flatten(1).mean(dim=1)
        restored_mae = F.l1_loss(restored, sharp, reduction='none').flatten(1).mean(dim=1)

        for index in range(blurry.size(0)):
            rows.append({
                'blurry_psnr': blurry_psnr[index].item(),
                'restored_psnr': restored_psnr[index].item(),
                'blurry_ssim': blurry_ssim[index].item(),
                'restored_ssim': restored_ssim[index].item(),
                'blurry_mae': blurry_mae[index].item(),
                'restored_mae': restored_mae[index].item(),
            })
    return pd.DataFrame(rows)


def plot_training_curves():
    history_path = RUN_ROOT / 'training_history.csv'
    if not history_path.exists():
        print('Training history not found. Skipping loss-curve figure.')
        return None

    history_frame = pd.read_csv(history_path)
    figure, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    axes[0].plot(history_frame['epoch'], history_frame['train_g_loss'], label='Generator')
    axes[0].plot(history_frame['epoch'], history_frame['train_d_loss'], label='Discriminator')
    axes[0].set_title('Generator and Discriminator Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history_frame['epoch'], history_frame['train_pixel_loss'], label='Pixel L1')
    axes[1].plot(history_frame['epoch'], history_frame['train_perceptual_loss'], label='Perceptual')
    axes[1].plot(history_frame['epoch'], history_frame['train_adv_loss'], label='Adversarial')
    axes[1].set_title('Generator Loss Components')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(history_frame['epoch'], history_frame['valid_l1'], color='tab:red')
    axes[2].set_title('Validation L1 Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('L1')
    axes[2].grid(alpha=0.3)

    axes[3].plot(history_frame['epoch'], history_frame['valid_psnr'], color='tab:green')
    axes[3].set_title('Validation PSNR')
    axes[3].set_xlabel('Epoch')
    axes[3].set_ylabel('PSNR (dB)')
    axes[3].grid(alpha=0.3)

    figure.tight_layout()
    output_path = EVALUATION_ROOT / 'deblurgan_training_curves.png'
    figure.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', output_path)
    return output_path


def plot_metric_summary(metrics_frame):
    summary = pd.DataFrame({
        'Metric': ['PSNR (dB)', 'SSIM', 'MAE'],
        'YOLO blurry crop': [metrics_frame['blurry_psnr'].mean(), metrics_frame['blurry_ssim'].mean(), metrics_frame['blurry_mae'].mean()],
        'DeblurGAN output': [metrics_frame['restored_psnr'].mean(), metrics_frame['restored_ssim'].mean(), metrics_frame['restored_mae'].mean()],
    })
    summary.to_csv(EVALUATION_ROOT / 'deblurgan_metric_summary.csv', index=False)
    metrics_frame.to_csv(EVALUATION_ROOT / 'deblurgan_per_sample_metrics.csv', index=False)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, metric, blurry_column, restored_column in zip(
        axes,
        ['PSNR (dB)', 'SSIM', 'MAE'],
        ['blurry_psnr', 'blurry_ssim', 'blurry_mae'],
        ['restored_psnr', 'restored_ssim', 'restored_mae'],
    ):
        values = [metrics_frame[blurry_column].mean(), metrics_frame[restored_column].mean()]
        axis.bar(['Blurry', 'Deblurred'], values, color=['tab:orange', 'tab:blue'])
        axis.set_title(metric)
        axis.grid(axis='y', alpha=0.3)
        for position, value in enumerate(values):
            axis.text(position, value, f'{value:.3f}', ha='center', va='bottom')

    figure.suptitle('DeblurGAN Restoration Metric Comparison')
    figure.tight_layout()
    output_path = EVALUATION_ROOT / 'deblurgan_metric_comparison.png'
    figure.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', output_path)
    display(summary)
    return output_path


@torch.no_grad()
def plot_qualitative_grid(generator_model, loader, count=5):
    generator_model.eval()
    blurry, sharp = next(iter(loader))
    blurry = blurry.to(TORCH_DEVICE, non_blocking=True)
    sharp = sharp.to(TORCH_DEVICE, non_blocking=True)
    restored = generator_model(blurry)
    count = min(count, blurry.size(0))

    figure, axes = plt.subplots(count, 4, figsize=(14, 3 * count))
    if count == 1:
        axes = np.expand_dims(axes, axis=0)

    for index in range(count):
        error = torch.abs(restored[index] - sharp[index]).mean(dim=0)
        panels = [
            (tensor_to_image(blurry[index]), 'YOLO blurry crop'),
            (tensor_to_image(restored[index]), 'DeblurGAN output'),
            (tensor_to_image(sharp[index]), 'Clean target'),
            (error.detach().cpu().numpy(), 'Absolute error'),
        ]
        for column, (image, title) in enumerate(panels):
            if column == 3:
                axes[index, column].imshow(image, cmap='magma', vmin=0, vmax=1)
            else:
                axes[index, column].imshow(image)
            axes[index, column].set_title(title)
            axes[index, column].axis('off')

    figure.tight_layout()
    output_path = EVALUATION_ROOT / 'deblurgan_qualitative_examples.png'
    figure.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', output_path)
    return output_path


def create_deblurgan_evaluation_figures():
    evaluation_generator = get_evaluation_generator()
    plot_training_curves()
    metrics_frame = collect_restoration_metrics(evaluation_generator, test_loader)
    plot_metric_summary(metrics_frame)
    plot_qualitative_grid(evaluation_generator, test_loader, count=5)
    print('Evaluation figures saved in:', EVALUATION_ROOT.resolve())


create_deblurgan_evaluation_figures()
